# Experiemnting with MF-BPR postive interaction matrix
To improve the results of the mf-bpr model, experiment  with only including positive interactions in the user-activity matrix. 

In [11]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

In [12]:
train_df = pd.read_csv(
    "../data/processed/train60.csv",
    dtype={
        "user_id":"string",
        "gmap_id":"string"
    }, 
    usecols=["user_id", "gmap_id", "rating"]
)

train_df = train_df[train_df["rating"] >= 4].copy()

train_df = train_df.drop_duplicates(["user_id", "gmap_id"])

In [13]:
user_indices, unique_users = pd.factorize(
    train_df["user_id"],
    sort=True
)

activity_indices, unique_activities = pd.factorize(
    train_df["gmap_id"],
    sort=True
)

user_to_index = {
    user_id: index
    for index, user_id in enumerate(unique_users)
}

activity_to_index = {
    activity_id: index
    for index, activity_id in enumerate(unique_activities)
}

index_to_user = {
    index: user_id
    for index, user_id in enumerate(unique_users)
}

index_to_activity = {
    index: activity_id
    for index, activity_id in enumerate(unique_activities)
}

num_users = len(unique_users)
num_activities = len(unique_activities)

In [14]:
user_activity_matrix = csr_matrix(
    (
        np.ones(len(train_df), dtype=np.float32),
        (user_indices, activity_indices)
    ),
    shape=(num_users, num_activities),
    dtype=np.float32
)

In [15]:
# Initialize the BPR model
bpr_model_60 = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.01,
    regularization=0.001,
    iterations=50,
    random_state=42
)

# Train using the sparse interaction matrix
bpr_model_60.fit(
    user_activity_matrix,
    show_progress=True
)

100%|██████████| 50/50 [02:21<00:00,  2.83s/it, train_auc=94.95%, skipped=0.15%]


In [16]:
def recommend_bpr(user_id, k=10):

    user_idx = user_to_index[user_id]

    recommended_items, scores = bpr_model_60.recommend(
        userid=user_idx,
        user_items=user_activity_matrix[user_idx],
        N=k,
        filter_already_liked_items=True
    )

    return recommended_items, scores

# Evaluation

In [17]:
import math

def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = len(set(recommended_k) & set(relevant))

    return hits / len(relevant)


def ndcg_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    # Actual DCG
    dcg = 0
    for i, item in enumerate(recommended_k):
        if item in relevant:
            dcg += 1 / math.log2(i + 2)

    # Best possible DCG
    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = set(recommended_k) & set(relevant)

    if len(hits) > 0:
        return 1
    else:
        return 0


def calculate_metrics(recommended, relevant, k=10):

    recall = recall_at_k(recommended, relevant, k)
    hit_rate = hit_rate_at_k(recommended, relevant, k)
    ndcg = ndcg_at_k(recommended, relevant, k)

    return {
        f"Recall@{k}": recall,
        f"HitRate@{k}": hit_rate,
        f"NDCG@{k}": ndcg
    }


In [18]:
sample_df = pd.read_csv(
    "../data/processed/validation_sample_1000_users.csv",
    dtype={"user_id": "string"}
)

validation_df = pd.read_csv(
    "../data/processed/validation20.csv",
    dtype={
        "user_id": "string",
        "gmap_id": "string"
    }
)

# Keep only positive validation interactions
validation_pos = validation_df[
    validation_df["rating"] >= 4
].copy()

# Build validation ground truth
relevant_by_user = (
    validation_pos
    .groupby("user_id")["gmap_id"]
    .apply(set)
    .to_dict()
)

# Use the SAME fixed 1,000 users
evaluation_users = sample_df["user_id"].tolist()

In [19]:
results = []

for user_id in evaluation_users:
    items, scores = recommend_bpr(user_id, k=10)

    recommended = [
        index_to_activity[item_idx]
        for item_idx in items
    ]

    metrics = calculate_metrics(
        recommended,
        relevant_by_user[user_id],
        k=10
    )

    results.append({
        "user_id": user_id,
        **metrics
    })

results_df = pd.DataFrame(results)

print("POSITIVE-ONLY MF-BPR")
print("Evaluated users:", len(results_df))

print(
    results_df[
        ["Recall@10", "HitRate@10", "NDCG@10"]
    ].mean()
)

POSITIVE-ONLY MF-BPR
Evaluated users: 1000
Recall@10     0.041734
HitRate@10    0.097000
NDCG@10       0.025614
dtype: float64


restricting training to only positive interactions did not significantly improve model performance, likely due to the skew of how positive the inetractions were in the dataset. 